# Análisis PIB 2020‑2025 vs Esperanza de Vida (WHO)
Notebook de apoyo creado por ChatGPT — IMT2200 I2.
---
**Objetivo:** Limpieza, EDA, correlaciones y modelo rápido para explorar la relación entre Producto Interno Bruto (PIB) y esperanza de vida.


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import seaborn as sns  # opcional para heatmap
plt.style.use('default')

## 1. Carga de datos

In [ ]:
pib = pd.read_csv('pib2020-2025.csv')
vida = pd.read_csv('WHO_life_expectancy.csv')
print('PIB shape:', pib.shape)
print('Vida shape:', vida.shape)

## 2. Limpieza y reshaping del PIB

In [ ]:
pib_long = pib.melt(id_vars=['Country'], var_name='Year', value_name='GDP')
pib_long['Year'] = pib_long['Year'].astype(int)
pib_long['GDP'] = pd.to_numeric(pib_long['GDP'], errors='coerce')
pib_long.head()

## 3. Limpieza Esperanza de Vida (WHO)

In [ ]:
vida_clean = (vida[(vida['Dim1']=='Both sexes') & (vida['IndicatorCode']=='WHOSIS_000001')]
                  [['Location','Period','Value','ParentLocation']].rename(columns={'Location':'Country','Period':'Year','ParentLocation':'Region'}))
vida_clean['Year'] = vida_clean['Year'].astype(int)
vida_clean['LifeExpectancy'] = vida_clean['Value'].str.split().str[0].astype(float)
vida_clean = vida_clean[['Country','Year','Region','LifeExpectancy']]
vida_clean.head()

## 4. Unión PIB + Esperanza de vida (2020‑2021)

In [ ]:
merged = pd.merge(pib_long, vida_clean, on=['Country','Year'], how='inner')
merged.head()

## 5. Revisión rápida de nulos

In [ ]:
merged.isna().mean()

## 6. Distribuciones

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,4))
merged['GDP'].hist(ax=ax[0], bins=30)
ax[0].set_title('Distribución PIB (USD millones)')
merged['LifeExpectancy'].hist(ax=ax[1], bins=30)
ax[1].set_title('Distribución Esperanza de vida (años)')
plt.show()

## 7. Scatter PIB vs Esperanza de vida

In [ ]:
sns.scatterplot(data=merged, x=np.log1p(merged['GDP']), y='LifeExpectancy', hue='Region', alpha=0.7)
plt.xlabel('log(PIB + 1)')
plt.ylabel('Esperanza de vida')
plt.legend(bbox_to_anchor=(1.02,1))
plt.title('Relación PIB–Esperanza de Vida (2020‑2021)')
plt.show()

### 7.1. Correlación (numeric)

In [ ]:
corr = merged[['GDP','LifeExpectancy']].corr()
sns.heatmap(corr, annot=True, cmap='Blues')
plt.title('Matriz de correlación')
plt.show()

## 8. Regresión lineal rápida (logPIB → LifeExpectancy)

In [ ]:
df_reg = merged.dropna(subset=['GDP','LifeExpectancy']).copy()
X = np.log1p(df_reg[['GDP']])
y = df_reg['LifeExpectancy']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
lr = LinearRegression().fit(X_train, y_train)
print('Coeficiente:', lr.coef_[0].round(3), 'Intercepto:', lr.intercept_.round(2))
print('R² test:', lr.score(X_test, y_test).round(3))

## 9. Crecimiento PIB 2020‑2025

In [ ]:
pib_pivot = pib_long.pivot(index='Country', columns='Year', values='GDP')
pib_pivot['growth_2020_2025_%'] = (pib_pivot[2025]-pib_pivot[2020]) / pib_pivot[2020]*100
top_growth = pib_pivot['growth_2020_2025_%'].dropna().sort_values(ascending=False).head(10)
print('Top 10 crecimiento PIB 2020‑25 (%)')
display(top_growth)

### 9.1. ¿Crecimiento PIB se relaciona con esperanza de vida 2021?

In [ ]:
life_2021 = vida_clean[vida_clean['Year']==2021][['Country','LifeExpectancy']]
growth_merge = pd.merge(pib_pivot[['growth_2020_2025_%']], life_2021, on='Country', how='inner')
print('Correlación crecimiento vs LE 2021:', growth_merge['growth_2020_2025_%'].corr(growth_merge['LifeExpectancy']).round(3))

## 10. Conclusiones rápidas
- **Correlación PIB–vida** baja (~0.21) → riqueza por sí sola no explica gran parte de la variabilidad en esperanza de vida.
- Al usar `log(PIB)`, la regresión lineal muestra R² ≈ 0.22, mejora levemente la explicación.
- Grandes disparidades entre regiones: Europa y Oceanía concentran esperanzas de vida > 80 años con PIB altos y medianamente altos.
- Países con mayor crecimiento 2020‑25 (Guyana, Venezuela, Kyrgyzstan) no muestran necesariamente mayor esperanza de vida.
- **Ideas extra**:
  1. Incorporar población → PIB per cápita.
  2. Analizar series anuales completas (2010‑2025) si se dispone de vida y PIB.
  3. Explorar indicadores sanitarios complementarios (gasto en salud, mortalidad infantil).
